# Detchar Clustering

A ideia deste código é encontrar grupos de triggers gerados pelo Omicron de forma independente. A estrutura da seção **Routine** é feita para que se salve dia-a-dia para não permitir a perda de dados caso o servidor crashe. Os caminhos usados devem seguir uma estrutura clara, para que o armazenamento seja automático com base nas variaveis declaradas na seção **Parameters**.

Caso ocorra algum erro, entre em contato com o responsável: **$\text{erick.sanches.123@gmail.ufrn.edu.br}$** e envie todo seu código e sua estrutura de pastas. Esta última pode ser obtida por **tree -L** através do terminal.

A arvore usada segue a seguinte estrutura:

```
Virgo
    triggers_total.csv
    gspyO3a.csv
    gspyO3b.csv
    
    zomicron
        // aqui ficam os arquivos .root da O3. Eles serão agrupados em "V1Data_processing.ipynb"
    zrepository
        // aqui ficam os intervalos de funcionamento do interferometro. Isso será usado para filtrar os triggeers.
        gwoscO3a.txt
        gwoscO3b.txt
        gwoscO4b.txt
    
        gdchar
            O3a
                // aqui serão armazenados os dados diários rodados em "Routine" para O3a do grupo detChar
            O3b
                // o mesmo para O3b
        gspy
            // aqui serão armazenados os dados diários rodados em "Routine" para O3a/O3b do Gravity Spy

    V0Clustering_optimzed.ipynb
        // clusterização do grupo DetChar
    V1Data_processing.ipynb
        // código para juntar .root's ou .csv's em um arquivo só
    V2Glitchgram_creation.ipynb
        // Dados clustering (glitches) e unclustering (triggers), devolve os vetores de SNR dos glitchgramas
    V3a_gdchar_tSNE.ipynb
        // roda o tSNE para O3a da clusterização do detchar
    V3a_tSNE.ipynb
        // toda o tSNE para O3a da clusterização do Gravity Spy
    V4HDBSCAN*.ipynb
        // estudos do HDBSCAN para classificação não-supervisionada
```

Formas de obtenção de dados:

- Os triggers foram obtidos através de um cluster Frances. Estão disponíveis no Drive do grupo Detchar
- Os intervalos de funcionamento do interferometro Virgo foram obtidos via **GWOSC**
- Também é possívei obter dados das corridas através do site **Zenodo**
- Os glitches clusterizados pelo Omicron/GravitySpy foram obtidos pelo cluster Italiano. Estão no mesmo Drive.

ps.: alguns **jupyterNotebook.ipynb** ou **arquivos.csv** não estão sendo usados e serão futuramente descartados.

## Libraries

In [20]:
import os
import pandas as pd
import numpy as np
import gc

from datetime import datetime, timedelta
from astropy.time import Time
import glob as glob

from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

## Functions

- **glitch_to_glitchgram:** constroi as matrizes dos glitchgramas a partir dos triggers e glitches
- **filter_intervals:** aplica o filtro nos triggers, considerando aqueles apenas durante o funcionamento OK do interferometro
- **clusterize_triggers1D/ clusterize_triggers2D:** funções de clusterização do grupo DetChar
- **run_routine:** rotina para rodar a clusterização dia-a-dia

In [21]:
def glitch_to_glitchgram(unclustered, clustered, deltat, 
                         fmin=3, fmax=8192, tbins=41, fbins=30):
    
    all_glitchgrams = []
    
    """
    This function create one glitchgram from unclustered Omicron data, for each glitch.

    Parameters:
        df_triggers (dataFrame): A dataFrame of GPStimes of all triggers inside of some window.
        tmin, tmax (numbers): limits of GPStimes that contain all triggers of some glitch.
        fmin, fmax (numbers): frequency range for the glitchgram.
        n_time_bins (int): number of time bins.
        n_freq_bins (int): number of frequency bins.
        norm (boolean): True if you need normalize SNR values.

    Returns:
        array for some glitch: each entry is a SNR value from flattened (binsf × binst) matrix.
    """

    times_all = unclustered['time'].values
    freqs_all = unclustered['frequency'].values
    snrs_all  = unclustered['snr'].values

    time_edges = np.linspace(-deltat/2, deltat/2, tbins+1)
    freq_edges = np.logspace(np.log10(fmin), np.log10(fmax), fbins+1)
    
    for i, row in enumerate(clustered.itertuples(index=False)):

        #t_center = getattr(row, 'GPStime')
        t_center = row.time
    
        tmin = t_center - deltat/2
        tmax = t_center + deltat/2
    
        i0 = np.searchsorted(times_all, tmin, side="left")
        i1 = np.searchsorted(times_all, tmax, side="right")
    
        times = times_all[i0:i1] - t_center
        freqs = freqs_all[i0:i1]
        snrs  = snrs_all[i0:i1]

        '''if len(times) < 10: # Equivalente ao tcut
            continue'''
    
        df_triggers = pd.DataFrame({'time': times,'frequency': freqs,'snr': snrs})
        
        t_idx = np.digitize(df_triggers['time'].values, time_edges) - 1
        f_idx = np.digitize(df_triggers['frequency'].values, freq_edges) - 1
            
        # initial empty matrix
        A = np.zeros((fbins, tbins), dtype=float)
    
        for ti, fi, si in zip(t_idx, f_idx, snrs):
            if 0 <= ti < tbins and 0 <= fi < fbins:
                if si > A[fi, ti]:
                    A[fi, ti] = si
                    
        '''# normalização do SNR 
        if len(snrs) > 0:
            snr_min = snrs.min()
            snr_max = snrs.max()
            if snr_max > snr_min:
                norm_snrs = (snrs - snr_min) / (snr_max - snr_min)
            else:
                norm_snrs = np.zeros_like(snrs)
        else:
            norm_snrs = snrs'''

        all_glitchgrams.append(A)
    
    return all_glitchgrams

In [22]:
def filter_intervals(gps_start, gps_end, intervals):
    intervals_crop = intervals[(intervals['int2'] >= gps_start) & (intervals['int1'] <= gps_end)].sort_values('int1')
    intervals_crop['int1'] = intervals_crop['int1'].astype('float64')
    
    total_triggers_winddow = 0
    filtered_chunks = []
    
    for chunk in pd.read_csv(base + '/triggers_total.csv', chunksize=10000000):
    
        chunk['time'] = chunk['time'].astype('float64')
        
        chunk = chunk[(chunk['time'] >= gps_start) & (chunk['time'] <= gps_end)].sort_values('time')
        if chunk.empty:
            continue
    
        total_triggers_winddow += len(chunk)
        
        merged = pd.merge_asof(chunk, intervals_crop, left_on='time', right_on='int1', direction='backward')
        valid = merged[(merged['time'] >= merged['int1']) & (merged['time'] <= merged['int2'])]

        if not valid.empty:
            filtered_chunks.append(valid[['time', 'frequency', 'snr']])
    if not filtered_chunks:
        return pd.DataFrame(columns=['time', 'frequency', 'snr'])
    
    triggers_crop = pd.concat(filtered_chunks, ignore_index=True)

    return triggers_crop

In [23]:
def clusterize_triggers1D(gltab, window, min_triggers):
    
    if len(gltab) == 0:
        return pd.DataFrame()

    table = gltab.copy()
    table = table.sort_values(by='time').reset_index(drop=True)

    times = table['time'].values
    deltas = times[1:] - times[:-1]

    new_cluster_mask = np.concatenate([[False], deltas > window])
    cluster_ids = np.cumsum(new_cluster_mask)

    table['cluster'] = cluster_ids

    # filter: Retains only clusters with "min_triggers" or more triggers
    counts = table['cluster'].value_counts()
    valid_clusters = counts[counts >= min_triggers].index
    
    table_filtered = table[table['cluster'].isin(valid_clusters)]

    if table_filtered.empty:
        return pd.DataFrame()
    
    # groups by cluster and extracts the representative (row with the highest SNR) + extremes
    summary_rows = []

    print(table_filtered)
    
    for cl, group in table_filtered.groupby('cluster'):
        # Encontra o trigger com maior SNR dentro deste grupo
        max_row = group.loc[group['snr'].idxmax()]

        summary_rows.append({
            'cluster': cl,
            'num_triggers': len(group), # quantidade de triggers no cluster
            'tstart': group['time'].min(),
            'tend': group['time'].max(),
            'time': max_row['time'],
            'frequency': max_row['frequency'],
            'min_freq': group['frequency'].min(),
            'max_freq': group['frequency'].max(),
            'snr': max_row['snr']
        })

    return pd.DataFrame(summary_rows)

In [24]:
def clusterize_triggers2D(gltab, window_T, window_F, min_triggers):

    if len(gltab) == 0:
        return pd.DataFrame()

    table = gltab.copy()

    times = table['time'].values
    freqs = table['frequency'].values
    n = len(times)

    # sort by time
    sort_idx = np.argsort(times)
    times_sorted = times[sort_idx]
    freqs_sorted = freqs[sort_idx]
    
    connections = []

    # for each trigger, look ONLY at the subsequent ones that fall within the time window
    for i in range(n):
        t_curr = times_sorted[i]
        f_curr = freqs_sorted[i]

        # searches only for candidates within the time window t_curr + window_T
        for j in range(i + 1, n):
            if times_sorted[j] - t_curr > window_T:
                break

            # tests the frequency condition
            if abs(freqs_sorted[j] - f_curr) /f_curr <= window_F:
                
                # connects trigger i to trigger j
                connections.append((sort_idx[i], sort_idx[j]))
    
    # form groupss based on the connections found
    if connections:
        pairs = np.array(connections)
        data = np.ones(len(pairs), dtype=bool)
        adj = csr_matrix((data, (pairs[:, 0], pairs[:, 1])), shape=(n, n))
        _, cluster_ids = connected_components(csgraph=adj, directed=False)
    else:
        cluster_ids = np.arange(n)

    table['cluster'] = cluster_ids

    # filter: Retains only clusters with "min_triggers" or more triggers
    counts = table['cluster'].value_counts()
    valid_clusters = counts[counts >= min_triggers].index
    
    table_filtered = table[table['cluster'].isin(valid_clusters)]

    if table_filtered.empty:
        return pd.DataFrame()
    
    # groups by cluster and extracts the representative (row with the highest SNR) + extremes
    summary_rows = []
    
    for cl, group in table_filtered.groupby('cluster'):
        # Encontra o trigger com maior SNR dentro deste grupo
        max_row = group.loc[group['snr'].idxmax()]

        summary_rows.append({
            'cluster': cl,
            'num_triggers': len(group), # quantidade de triggers no cluster
            'tstart': group['time'].min(),
            'tend': group['time'].max(),
            'time': max_row['time'],
            'frequency': max_row['frequency'],
            'min_freq': group['frequency'].min(),
            'max_freq': group['frequency'].max(),
            'snr': max_row['snr']
        })

    return pd.DataFrame(summary_rows)

In [25]:
def run_routine(current_day, end_day, intervals, min_triggers, output_dir, global_cluster_ID=0):

    # creates the folder to save the daily records, if it doesn't existt
    os.makedirs(output_dir, exist_ok=True)
    
    # clustered_list = []
    trigger_count = []
    glitch_count = []

    '''gps_start_filt = Time(current_day, scale="utc").gps
    gps_end_filt = Time(end_day, scale="utc").gps

    triggers_crop = filter_intervals(gps_start_filt, gps_end_filt, intervals)'''
    
    while current_day <= end_day:

        date_str = current_day.strftime('%Y-%m-%d')
        out_file = os.path.join(output_dir, f"glitches_{date_str}.parquet")

        # skip if the day has already been processed and saved
        if os.path.exists(out_file):
            print(f"Day {date_str} already processed. Skipping...")
            current_day += timedelta(days=1)
            continue
        
        # declare the start and end of the current day
        day_start = datetime(current_day.year, current_day.month, current_day.day, 0, 0, 0)
        day_end   = datetime(current_day.year, current_day.month, current_day.day, 23, 59, 59)
    
        print(f"Processing day: {current_day.date()}")
    
        # convert the time to gpsTIME
        gps_start = Time(day_start, scale="utc").gps
        gps_end   = Time(day_end, scale="utc").gps

        # loads and filters only the triggers for the current day
        gltab = filter_intervals(gps_start, gps_end, intervals)
        
        '''
        # limit the intervals to the current day only
        day_interval = intervals[(intervals['int2'] >= gps_start) & (intervals['int1'] <= gps_end)].copy()
        
        # cut intervals before gps_start and after gps_end
        day_interval['start_recortado'] = day_interval['int1'].clip(lower=gps_start)
        day_interval['end_recortado']   = day_interval['int2'].clip(upper=gps_end)
    
        # duration of the intervals of the current day
        day_duration_sec = (day_interval['end_recortado'] - day_interval['start_recortado']).sum()
        day_duration_hrs = day_duration_sec / 3600
    
        print(f'Opening hours for this day: {day_duration_hrs:.2f}')
    
        # triggers of the current day
        trigfiles = triggers_crop[(triggers_crop['time'] >= gps_start) & (triggers_crop['time'] <= gps_end)]
        gltab = trigfiles.copy()

        del trigfiles, day_interval
        '''
        
        num_trigggers = len(gltab)
        trigger_count.append(num_trigggers)
        print(f'Triggers count for this day: {num_trigggers}\n')
    
        # CLUSTERING OF THE CURRENT DAY
        if not gltab.empty:
            gltab = gltab.sort_values(by="time").drop_duplicates(subset=["time"]).reset_index(drop=True)
    
            # the MAIN FUNCTION of this clustering process
            if clust_dim == 1:
                clustered = clusterize_triggers1D(gltab, temp_dist, min_triggers)
            else:
                clustered = clusterize_triggers2D(gltab, temp_dist, freq_dist, min_triggers)

            num_glitches = len(clustered)
            glitch_count.append(num_glitches)
            print(f'\nGlitches count for this day: {num_glitches}\n')
            
            # adds the day's result to the list if there are glitches
            if not clustered.empty:
                
                # adjusts the day's IDs by adding the accumulated ID
                clustered['cluster'] += global_cluster_ID
    
                # update the ID for the next day to avoid repeating numbers
                global_cluster_ID = clustered['cluster'].max() + 1

                abs_out_file = os.path.abspath(out_file)

                try:
                    # salva o dia no disco
                    clustered.to_parquet(abs_out_file, index=False)
                    print(f"Successfully saved: {out_file}")
                except Exception as e:
                    csv_file = abs_out_file.replace('.parquet', '.csv')
                    clustered.to_csv(csv_file, index=False)
                    print(f"parquet failed ({e}). SAVED IN CSV in: {csv_file}")
                # clustered_list.append(clustered)

                del clustered
        else:
            glitch_count.append(0)
            print("Glitches count for this day: 0")

        del gltab
        
        # forces the release of unused RAM
        gc.collect()
        
        current_day += timedelta(days=1)

    """
    del triggers_crop
    gc.collect()
    """

    # final concatenation reading files from the disk
    print("Concatenating results saved to disk...")
    saved_files = sorted(glob.glob(os.path.join(output_dir, "glitches_*.parquet")))

    if saved_files:
        df_list = [pd.read_parquet(f) for f in saved_files]
        df_glitches_final = pd.concat(df_list, ignore_index=True)
        del df_list
        gc.collect()
    else:
        df_glitches_final = pd.DataFrame()

    return df_glitches_final, trigger_count, glitch_count

    # appends to the FINAL DataFrame every day
    """
    if clustered_list:
        df_glitches_final = pd.concat(clustered_list, ignore_index=True)
    else:
        df_glitches_final = pd.DataFrame()

    return df_glitches_final
    """

In [26]:
def plot_grid(arr, title):
    n = len(arr)
    if n == 0:
        print(f"Nenhum glitch encontrado em {title}")
        return
    
    cols = 5
    rows = int(np.ceil(n / cols))
    
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 2))
    axes = np.atleast_1d(axes).flatten()
    
    fig.suptitle(f"{title} (Total: {n})", fontsize=12, fontweight='bold')
    
    for i in range(n):
        axes[i].imshow(arr[i], origin='lower', aspect='auto')
        axes[i].axis('off')
        
    for j in range(n, len(axes)):
        axes[j].axis('off')
        
    plt.tight_layout()
    plt.show()

## Parameters

- (int) **min_triggers:** corte da quantidade mínima de triggers por glitch
- (float) **freq_dist:** porcentagem de corte na frequencia
- (int) **clust_dim:** tipo de clusterização testada (1D ou 2D)
- (float) **temp_dist:** distancia temporal minima entre 2 triggers para pertencerem ao mesmo glitch
- (bool) **filterr:** se os glitches serão filtrados em uma quantidade de 10k ou não
- (string) **clust, run:** grupo de clusterização (gdchar ou gspy) e corrida (O3a ou O3b)
- **sufixo:** serve apenas para diferenciar o nome do arquivo no salvamento automatico

Ao preencher todos esses dados e os caminhos **base**, **repository** e **daily_clustering**, o código pode ser executado por completo sem se preocupar com nomeação de arquivos ou caminhos de salvamento.

In [27]:
# Variables:
min_triggers = 20
freq_dist = 0.2

# Parameters:
clust_dim = 1
temp_dist = 0.1
deltat = 2.0
filterr = False

clust, run = 'gdchar', 'O3a'
sufixo = '_filtered_' if filterr else '_'

In [28]:
base = os.getcwd()
repository = base + '/zrepository'
temp = repository + '/' + clust + '/' + run + '/' + str(clust_dim) + 'D_dailyClustering_c' + str(min_triggers)

daily_clustering = (temp + '_f' + str(freq_dist) if clust_dim == 2 else temp)

In [29]:
base

'/home/esanches/Projetos/tSNE/Virgo'

In [30]:
repository

'/home/esanches/Projetos/tSNE/Virgo/zrepository'

In [31]:
daily_clustering

'/home/esanches/Projetos/tSNE/Virgo/zrepository/gdchar/O3a/1D_dailyClustering_c20'

The **base** will be used to access the triggers, **repository** to access the operating intervals of Virgo interferometer and **daily_clustering** to access the daily clustering results, obviously.

## Routine

Se por acaso o código parar por algum motivo, reinicie o servidor e veja em **Virgo/zrepository/group/run/daily_clustering/** em qual dia parou. Depois, mude o **start_day_O3a** ou **start_day_O3b** para este dia e continue normalmente de onde tinha parado.

In [33]:
start_day_O3a = datetime(2019, 4, 1, 0, 0, 0)
end_day_O3a   = datetime(2019, 10, 1, 15, 0, 0)

# start_day_O3a = datetime(2019, 9, 15, 0, 0, 0)

start_day_O3b = datetime(2019, 11, 1, 15, 0, 0)
end_day_O3b   = datetime(2020, 3, 27, 17, 0, 0)

# If you want to convert to gps time: 

# gps_start = Time(start_day_O3b, scale="utc").gps
# gps_end = Time(end_day_O3b, scale="utc").gps

In [34]:
start_days = {'O3a': start_day_O3a, 'O3b': start_day_O3b}
end_days = {'O3a': end_day_O3a, 'O3b': end_day_O3b}

start = start_days[run]
end = end_days[run]

In [35]:
intervals = pd.read_csv(repository + '/gwosc' + run + '.txt', sep=' ', header=None, names=['int1', 'int2', 'duration'])

intervals_hour = intervals['duration'].values.sum() / 3600
intervals_hour

np.float64(3344.1469444444447)

These result represent the operation time of the interferometer during these **run**

### Run the Routine ~ It might take some time

In [ ]:
df_glitches_final = run_routine(start, end, intervals, min_triggers, output_dir=daily_clustering)

Processing day: 2019-04-01


## Combine and save the .parquet files

These will put together the data obtained from Routine in only one .csv arquive

In [ ]:
if clust_dim == 1:
    arq_name = 'clustered' + sufixo + 'c' + str(min_triggers) + '.csv'
else:
    arq_name = 'clustered' + sufixo + 'c' + str(min_triggers) + '_f' + str(window_F) + '.csv'

arquivo_csv_saida = repository + '/' + clust + '/' + run + '/' + arq_name
arquivo_csv_saida

In [ ]:
arquivos_parquet = sorted(glob.glob(os.path.join(daily_clustering, "glitches_*.parquet")))

print(f"reading {len(arquivos_parquet)} daily files...")

df_list = [pd.read_parquet(f) for f in arquivos_parquet]
df_glitches_final = pd.concat(df_list, ignore_index=True)

df_glitches_final.to_csv(arquivo_csv_saida, index=False)

print(f"completed! Total of {len(df_glitches_final):,} glitches saved in:")
print(os.path.abspath(arquivo_csv_saida))

## Glitches per hour

calcular os intervals da O3a para ver o glitch per hour V

In [112]:
base

'/home/esanches/Projetos/tSNE/Virgo'

In [113]:
intervals = pd.read_csv(base + '/zrepository/gwosc' + run + '.txt', sep=' ', header=None, names=['int1', 'int2', 'duration'])
intervals_hour = intervals['duration'].values.sum() / 3600
intervals_hour

np.float64(3344.1469444444447)

In [129]:
repository

'/home/esanches/Projetos/tSNE/Virgo/zrepository'

In [138]:
glitches = pd.read_csv(repository + '/' + clust + '/' + run + '/' + 'clustered' + sufixo + cut + '.csv')
len(glitches)

80769

In [137]:
glitch_per_hour = len(glitches) / intervals_hour
glitch_per_hour

np.float64(24.15234777113479)